In [4]:
# 1. Install the specialized installer
!pip install torch_geometric

# 2. Use the installer to get the optional dependencies
# This automatically handles the version matching for pyg-lib, etc.
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 14.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 41.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 116.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 37.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.2 MB/s eta 0:00:00


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import APPNP
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

cora     = Planetoid(root='/tmp/Cora',     name='Cora')
citeseer = Planetoid(root='/tmp/Citeseer', name='Citeseer')
data_cora     = cora[0].to(device)
data_citeseer = citeseer[0].to(device)

Device: cuda


# **Two-step of APPNP**

Step 1: MLP transforms features → initial predictions

Step 2: APPNP propagates predictions using PageRank formula

# PageRank propagation for K steps:
  Z^(0) = H          (MLP output)
  Z^(k) = (1-α)·Â·Z^(k-1) + α·H

  α = teleport probability (how much to keep own prediction)
  K = number of propagation steps


In [2]:
from torch_geometric.utils import add_self_loops, degree
from torch_sparse import SparseTensor, matmul

edge_index = data_cora.edge_index
num_nodes  = data_cora.num_nodes
ei_sl, _   = add_self_loops(edge_index, num_nodes=num_nodes)
row, col   = ei_sl
deg        = degree(col, num_nodes=num_nodes)
norm       = deg[row].pow(-0.5) * deg[col].pow(-0.5)
adj        = SparseTensor(row=row, col=col, value=norm,
                           sparse_sizes=(num_nodes, num_nodes))

# Simulate APPNP propagation with α=0.1, K=3
alpha = 0.1
K     = 3
H     = data_cora.x.float()   # pretend MLP output = raw features
Z     = H.clone()

print("APPNP propagation simulation (α=0.1, K=3):")
print(f"Step 0 (MLP output H) — node 0, first 5 values: {H[0,:5].tolist()}")

for k in range(1, K+1):
    Z = (1 - alpha) * matmul(adj, Z) + alpha * H
    print(f"Step {k} — node 0, first 5 values: {Z[0,:5].tolist()}")

APPNP propagation simulation (α=0.1, K=3):
Step 0 (MLP output H) — node 0, first 5 values: [0.0, 0.0, 0.0, 0.0, 0.0]
Step 1 — node 0, first 5 values: [0.0, 0.0, 0.0, 0.0, 0.0]
Step 2 — node 0, first 5 values: [0.0, 0.0, 0.0, 0.0, 0.0]
Step 3 — node 0, first 5 values: [0.001262664794921875, 0.0021555032581090927, 0.003857232630252838, 0.002809107070788741, 0.0]


# **APPNP model**

In [3]:
class APPNPNet(nn.Module):
    def __init__(self, in_channels, hidden_channels,
                 out_channels, K=10, alpha=0.1, dropout=0.5):
        super().__init__()
        self.dropout = dropout

        # Step 1: MLP — two linear layers, no graph involved
        self.lin1 = nn.Linear(in_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, out_channels)

        # Step 2: APPNP propagation — no learnable parameters
        # K=10 propagation steps, α=0.1 teleport probability
        self.prop = APPNP(K=K, alpha=alpha)

    def forward(self, x, edge_index):
        # --- Step 1: MLP (feature transformation) ---
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)           # shape: [N, num_classes]

        # --- Step 2: Propagation (spread predictions) ---
        # No weights learned here — pure graph diffusion
        x = self.prop(x, edge_index)
        return x

def make_appnp(dataset, K=10, alpha=0.1):
    return APPNPNet(
        in_channels     = dataset.num_node_features,
        hidden_channels = 64,
        out_channels    = dataset.num_classes,
        K               = K,
        alpha           = alpha
    ).to(device)

model_cora = make_appnp(cora)
print(model_cora)
print(f"\nLearnable parameters : {sum(p.numel() for p in model_cora.parameters()):,}")

APPNPNet(
  (lin1): Linear(in_features=1433, out_features=64, bias=True)
  (lin2): Linear(in_features=64, out_features=7, bias=True)
  (prop): APPNP(K=10, alpha=0.1)
)

Learnable parameters : 92,231


# **Before training snapshot**


In [4]:
model_cora.eval()
with torch.no_grad():
    out_before  = model_cora(data_cora.x, data_cora.edge_index)
    pred_before = out_before.argmax(dim=1)

acc_before = (pred_before[data_cora.test_mask] ==
              data_cora.y[data_cora.test_mask])
acc_before = acc_before.sum().item() / data_cora.test_mask.sum().item()
print(f"Accuracy BEFORE training : {acc_before*100:.1f}%")
print(f"Random chance baseline   : {100/7:.1f}%")
test_indices = data_cora.test_mask.nonzero(as_tuple=True)[0][:10]


Accuracy BEFORE training : 9.0%
Random chance baseline   : 14.3%


# **Train on Cora and Citeseer**

In [5]:
def train(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask],
                             data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        out  = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        acc  = (pred[mask] == data.y[mask]).sum() / mask.sum()
    return acc.item()

def run_experiment(model, data, dataset_name,
                    lr=0.01, weight_decay=5e-4, epochs=200):
    optimizer    = torch.optim.Adam(model.parameters(),
                                     lr=lr,
                                     weight_decay=weight_decay)
    train_losses = []
    val_accs     = []

    for epoch in range(1, epochs + 1):
        loss    = train(model, data, optimizer)
        val_acc = evaluate(model, data, data.val_mask)
        train_losses.append(loss)
        val_accs.append(val_acc)
        if epoch % 40 == 0:
            print(f"[{dataset_name}] Epoch {epoch:03d} | "
                  f"Loss: {loss:.4f} | Val Acc: {val_acc:.4f}")

    test_acc = evaluate(model, data, data.test_mask)
    print(f"\n[{dataset_name}] Test accuracy: {test_acc*100:.1f}%\n")
    return train_losses, val_accs, test_acc

# Running on Cora
losses_cora, vaccs_cora, test_cora = run_experiment(
    model_cora, data_cora, "Cora"
)

# Running on Citeseer
model_citeseer = make_appnp(citeseer)
losses_cs, vaccs_cs, test_cs = run_experiment(
    model_citeseer, data_citeseer, "Citeseer"
)

[Cora] Epoch 040 | Loss: 0.0248 | Val Acc: 0.7840
[Cora] Epoch 080 | Loss: 0.0395 | Val Acc: 0.7800
[Cora] Epoch 120 | Loss: 0.0301 | Val Acc: 0.7800
[Cora] Epoch 160 | Loss: 0.0248 | Val Acc: 0.7780
[Cora] Epoch 200 | Loss: 0.0243 | Val Acc: 0.7740

[Cora] Test accuracy: 79.6%

[Citeseer] Epoch 040 | Loss: 0.0422 | Val Acc: 0.6720
[Citeseer] Epoch 080 | Loss: 0.0397 | Val Acc: 0.7080
[Citeseer] Epoch 120 | Loss: 0.0286 | Val Acc: 0.6880
[Citeseer] Epoch 160 | Loss: 0.0357 | Val Acc: 0.6940
[Citeseer] Epoch 200 | Loss: 0.0394 | Val Acc: 0.6800

[Citeseer] Test accuracy: 66.8%



# **Effect of K and alpha on Cora**

- K controls how many hops to propagate

- alpha controls how much to retain own prediction

In [6]:
print("Effect of K (propagation steps) — alpha fixed at 0.1:")
print(f"{'K':>5} {'Test Acc':>10}")
print("-" * 18)
for K in [2, 5, 10, 20]:
    m = make_appnp(cora, K=K, alpha=0.1)
    run_experiment(m, data_cora, f"K={K}",
                   epochs=200)
    acc = evaluate(m, data_cora, data_cora.test_mask)
    print(f"{K:>5} {acc*100:>9.1f}%")

print("\nEffect of alpha (teleport) — K fixed at 10:")
print(f"{'alpha':>7} {'Test Acc':>10}")
print("-" * 20)
for alpha in [0.05, 0.1, 0.2, 0.5]:
    m = make_appnp(cora, K=10, alpha=alpha)
    run_experiment(m, data_cora, f"α={alpha}",
                   epochs=200)
    acc = evaluate(m, data_cora, data_cora.test_mask)
    print(f"{alpha:>7} {acc*100:>9.1f}%")

Effect of K (propagation steps) — alpha fixed at 0.1:
    K   Test Acc
------------------
[K=2] Epoch 040 | Loss: 0.0174 | Val Acc: 0.7660
[K=2] Epoch 080 | Loss: 0.0218 | Val Acc: 0.7840
[K=2] Epoch 120 | Loss: 0.0239 | Val Acc: 0.7820
[K=2] Epoch 160 | Loss: 0.0283 | Val Acc: 0.7820
[K=2] Epoch 200 | Loss: 0.0181 | Val Acc: 0.7620

[K=2] Test accuracy: 80.7%

    2      80.7%
[K=5] Epoch 040 | Loss: 0.0327 | Val Acc: 0.7780
[K=5] Epoch 080 | Loss: 0.0442 | Val Acc: 0.7920
[K=5] Epoch 120 | Loss: 0.0437 | Val Acc: 0.7800
[K=5] Epoch 160 | Loss: 0.0223 | Val Acc: 0.7740
[K=5] Epoch 200 | Loss: 0.0214 | Val Acc: 0.7820

[K=5] Test accuracy: 81.3%

    5      81.3%
[K=10] Epoch 040 | Loss: 0.0352 | Val Acc: 0.7800
[K=10] Epoch 080 | Loss: 0.0237 | Val Acc: 0.7900
[K=10] Epoch 120 | Loss: 0.0244 | Val Acc: 0.7800
[K=10] Epoch 160 | Loss: 0.0244 | Val Acc: 0.7900
[K=10] Epoch 200 | Loss: 0.0256 | Val Acc: 0.7920

[K=10] Test accuracy: 80.0%

   10      80.0%
[K=20] Epoch 040 | Loss: 0.0290